# Simulation Walkthrough

This notebook runs the default express/local crossing scenario tick by tick. It is meant for inspection, debugging, and learning the engine.

In [1]:
import os
import sys

repo_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from scenarios.scenario_1 import build_trains
from src.railway.network import default_network
from src.planning.scheduler import Scheduler
from src.simulation.simulator import Simulator
from src.planning.actions import ActionType


In [2]:
trains = build_trains()
sim = Simulator(trains, Scheduler(), network=default_network)

def train_table(sim):
    return [
        {
            "name": train.name,
            "station": sim.network.station_name(train.current_station),
            "line": train.line,
            "destination": sim.network.station_name(train.destination_station),
            "finished": train.finished,
            "waiting_time": train.waiting_time,
            "loop_entries": train.loop_entries,
        }
        for train in sim.trains
    ]

train_table(sim)


[{'name': 'EXP1',
  'station': 'Virar',
  'line': 'main',
  'destination': 'Dadar',
  'finished': False,
  'waiting_time': 0,
  'loop_entries': 0},
 {'name': 'LOC1',
  'station': 'Dadar',
  'line': 'main',
  'destination': 'Virar',
  'finished': False,
  'waiting_time': 0,
  'loop_entries': 0}]

In [3]:
print(sim.network.stations)

[Station(name='Virar', has_loop=False, platform_count=1), Station(name='Bhayandar', has_loop=False, platform_count=1), Station(name='Borivali', has_loop=True, platform_count=1), Station(name='Andheri', has_loop=True, platform_count=1), Station(name='Bandra', has_loop=False, platform_count=1), Station(name='Dadar', has_loop=False, platform_count=1)]


In [4]:
print(sim.network.next_station_for(trains[1]))

4


In [5]:
print(sim.occupancy_state.station_lines)

{(0, 'main'): 'EXP1', (1, 'main'): None, (2, 'main'): None, (2, 'loop'): None, (3, 'main'): None, (3, 'loop'): None, (4, 'main'): None, (5, 'main'): 'LOC1'}


In [6]:
actions = sim.step()
[
    {
        "train": action.train_name,
        "action": action.action_type.value,
        "reason": action.reason,
        "block": action.block,
        "conflict": action.conflict,
    }
    for action in actions
]



TIME 0
EXP1: Virar -> Bhayandar (advance toward destination)
LOC1: Dadar -> Bandra (advance toward destination)


[{'train': 'EXP1',
  'action': 'MOVE',
  'reason': 'advance toward destination',
  'block': (0, 1),
  'conflict': False},
 {'train': 'LOC1',
  'action': 'MOVE',
  'reason': 'advance toward destination',
  'block': (4, 5),
  'conflict': False}]

In [21]:
train_table(sim)

[{'name': 'EXP1',
  'station': 'Bhayandar',
  'track': 'main',
  'destination': 'Dadar',
  'finished': False,
  'waiting_time': 0,
  'loop_entries': 0},
 {'name': 'LOC1',
  'station': 'Bandra',
  'track': 'main',
  'destination': 'Virar',
  'finished': False,
  'waiting_time': 0,
  'loop_entries': 0}]

In [7]:
while sim.active_trains():
    sim.step()

sim.get_metrics()



TIME 1
EXP1: Bhayandar -> Borivali (advance toward destination)
LOC1: Bandra -> Andheri (advance toward destination)

TIME 2
LOC1: Andheri main -> loop (enter loop at Andheri to allow EXP1 to cross)
EXP1: Borivali -> Andheri (advance toward destination)

TIME 3
EXP1: Andheri -> Bandra (advance toward destination)
LOC1: Andheri loop -> main (exit loop after crossing path is clear)

TIME 4
EXP1: Bandra -> Dadar arrived (move into destination and leave controlled section)
LOC1: Andheri -> Borivali (advance toward destination)

TIME 5
LOC1: Borivali -> Bhayandar (advance toward destination)

TIME 6
LOC1: Bhayandar -> Virar arrived (move into destination and leave controlled section)


{'total_ticks': 7,
 'arrived_trains': 2,
 'active_trains': 0,
 'throughput': 0.2857142857142857,
 'conflict_count': 0,
 'loop_usage': 1,
 'trains': {'EXP1': {'finished': True,
   'waiting_time': 0,
   'completion_time': 5,
   'current_station': 'Dadar',
   'line': 'main',
   'loop_entries': 0},
  'LOC1': {'finished': True,
   'waiting_time': 0,
   'completion_time': 7,
   'current_station': 'Virar',
   'line': 'main',
   'loop_entries': 1}}}

In [8]:
timeline = []
for tick in sim.history:
    for action in tick["actions"]:
        timeline.append(
            {
                "time": tick["time"],
                "train": action.train_name,
                "action": action.action_type.value,
                "reason": action.reason,
                "block": action.block,
                "conflict": action.conflict,
            }
        )

timeline


[{'time': 0,
  'train': 'EXP1',
  'action': 'MOVE',
  'reason': 'advance toward destination',
  'block': (0, 1),
  'conflict': False},
 {'time': 0,
  'train': 'LOC1',
  'action': 'MOVE',
  'reason': 'advance toward destination',
  'block': (4, 5),
  'conflict': False},
 {'time': 1,
  'train': 'EXP1',
  'action': 'MOVE',
  'reason': 'advance toward destination',
  'block': (1, 2),
  'conflict': False},
 {'time': 1,
  'train': 'LOC1',
  'action': 'MOVE',
  'reason': 'advance toward destination',
  'block': (3, 4),
  'conflict': False},
 {'time': 2,
  'train': 'LOC1',
  'action': 'ENTER_LOOP',
  'reason': 'enter loop at Andheri to allow EXP1 to cross',
  'block': None,
  'conflict': False},
 {'time': 2,
  'train': 'EXP1',
  'action': 'MOVE',
  'reason': 'advance toward destination',
  'block': (2, 3),
  'conflict': False},
 {'time': 3,
  'train': 'EXP1',
  'action': 'MOVE',
  'reason': 'advance toward destination',
  'block': (3, 4),
  'conflict': False},
 {'time': 3,
  'train': 'LOC1',
 